# Week 13: Reinforcement Learning & RLHF
## Notebook 09: Soft Actor-Critic (SAC) & Maximum Entropy RL

### 1. Conceptual Intuition & Theoretical Background
Standard RL algorithms maximize expected cumulative reward: $\mathbb{E}[\sum r_t]$. This often causes policies to collapse into deterministic local optima.

**Maximum Entropy RL Framework (Ziebart, 2010; Haarnoja et al., 2018):**
Augments the objective by maximizing both expected reward **and policy entropy**:
$$J(\pi) = \sum_{t=0}^{T} \mathbb{E}_{(s_t, a_t) \sim \rho_\pi} \left[ r(s_t, a_t) + \alpha \mathcal{H}(\pi(\cdot \mid s_t)) \right]$$
Where $\mathcal{H}(\pi) = -\mathbb{E}_{a \sim \pi}[\ln \pi(a \mid s)]$ is the policy entropy, and $\alpha$ is the temperature parameter.

**Core Benefits:**
1. **Exploration:** Prevents premature policy collapse.
2. **Robustness:** Learns multi-modal policies capable of finding alternative solutions if the primary strategy is blocked.
3. **Clipped Double-Q:** Uses two critics ($Q_1, Q_2$) and takes their minimum to prevent overestimation bias.

---

### 2. Scratch Python Implementation: Soft Actor-Critic Objective

In [ ]:
import numpy as np

def soft_q_target(rewards, dones, next_states, critic1_fn, critic2_fn, actor_fn, alpha=0.2, gamma=0.99):
    """
    Computes SAC soft Bellman target with entropy regularization:
    y = r + gamma * (1 - done) * [ min(Q1, Q2) - alpha * log(pi(a|s')) ]
    """
    # Sample next actions and log-probabilities from current policy
    next_actions, next_log_probs = actor_fn(next_states)
    
    # Clipped Double-Q evaluation
    q1_next = critic1_fn(next_states, next_actions)
    q2_next = critic2_fn(next_states, next_actions)
    min_q_next = np.minimum(q1_next, q2_next)
    
    # Soft target evaluation with entropy bonus
    soft_v_next = min_q_next - alpha * next_log_probs
    targets = rewards + gamma * (1.0 - dones) * soft_v_next
    return targets

# Simulation: Continuous 1D Robotic Arm Environment
batch_size = 5
rewards = np.array([1.0, 0.5, 2.0, 0.0, -1.0])
dones = np.array([0, 0, 0, 0, 1])

# Mock Critic and Actor functions
def mock_critic1(s, a): return np.array([2.5, 1.8, 4.0, 0.5, 1.0])
def mock_critic2(s, a): return np.array([2.8, 1.6, 4.2, 0.4, 0.8])
def mock_actor(s): 
    actions = np.random.uniform(-1, 1, len(s))
    # Gaussian log-prob simulation
    log_probs = np.array([-0.8, -1.2, -0.5, -2.0, -0.9])
    return actions, log_probs

targets = soft_q_target(rewards, dones, np.zeros((batch_size, 2)), 
                        mock_critic1, mock_critic2, mock_actor, alpha=0.2)

print("--- Soft Actor-Critic (SAC) Target Formulation ---")
print(f"Observed Rewards:        {rewards}")
print(f"Clipped Min Q-Values:    {np.minimum(mock_critic1(0,0), mock_critic2(0,0))}")
print(f"Computed Soft Q-Targets: {np.round(targets, 3)}")
print("Entropy bonus encourages continuous action space exploration!")

### 3. Step-by-Step Code Walkthrough
1. **Entropy Regularization:** Subtracts $\alpha \ln \pi(a \mid s)$ from the Q-target to reward exploration.
2. **Clipped Double-Q:** Takes the minimum of two independent critics to prevent overestimation.
3. **Terminal Masking:** Drops future values when $d=1$.

---

### 4. Industry Applications & Trade-offs
- **Continuous Control Robotics:** Standard baseline algorithm for quadruped locomotion and robotic arm manipulation.
- **Trade-off:** Sensitive to the entropy temperature parameter $\alpha$ (often resolved via automated temperature tuning).

---

### 5. Top Interview Questions & Answers
**Q1: Why does Soft Actor-Critic learn multiple solutions to the same task?**
> **Answer:** Maximizing entropy forces the policy to assign non-zero probability mass to all actions that achieve near-optimal rewards, rather than collapsing to a single deterministic action.

**Q2: How does automated entropy temperature tuning work in SAC?**
> **Answer:** It formulates an optimization problem that adjusts $\alpha$ via gradient descent to match a target entropy $\bar{\mathcal{H}}$ (typically $-\dim(\mathcal{A})$).